In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# initial data processing
#   cascading the sentence number down to the sentences below it
#   necessary for dividing into training and testing sets
raw_dataset = pd.read_csv('ner_dataset2.csv', na_filter=False, dtype=str)
raw_dataset['Sentence Start'] = ~(raw_dataset['Sentence #'] == '')
raw_dataset['Sentence #'] = raw_dataset['Sentence #'].str.extract(r'(\d+)', expand=False).ffill().astype('int64')

In [3]:
def generate_ngram_model(training_set, gram_size=4):
    # ---------------- STAGE 0: PREPROCESSING HISTORY ----------------
    modified = pd.DataFrame(training_set)

    for n in range(gram_size - 1):
        modified[f'Previous POS {n + 1}'] = modified['POS'].shift(n + 1)
        modified[f'Previous POS {n + 1}'] = modified[f'Previous POS {n + 1}'].where(modified['Sentence #'] == modified['Sentence #'].shift(n + 1))

    # ---------------- STAGE 1: INITIAL COUNTING ----------------

    # the keys of this dictionary are effectively a path to the occurence
    #   for a unigram, this would be simply the current word
    #   for higher n-grams, this would be the sequence of previous POS with the current word
    # the values of each dictionary are also dictionaries
    #   the key is the observed type
    #   the value is the number of times this POS occurred
    ngram_map = {}

    def write_to_map(row):
        previous_pos = list(map(lambda n : row[f'Previous POS {n}'], list(range(1, gram_size))[::-1]))
        previous_pos = list(filter(lambda pos : not pd.isna(pos), previous_pos))
        
        if len(previous_pos) > 0:
            pattern = (*previous_pos, row['Word'])
        else:
            pattern = row['Word']

        if pattern in ngram_map:
            if row['POS'] in ngram_map[pattern]:
                ngram_map[pattern][row['POS']] += 1
            else:
                ngram_map[pattern][row['POS']] = 1
        else:
            ngram_map[pattern] = { row['POS']: 1 }

    modified.apply(write_to_map, axis=1)

    # ---------------- STAGE 2: FLATTENING ----------------

    # index 0 is the unanimous map, index 1 is the highest probability map
    ngram_model = [{}, {}]
    # iterate over every key, and store it in either the unanimous or non unanimous layer
    for (key, pos_map) in ngram_map.items():
        # check if key should be added to the unanimous map or the highest probability map
        ngram_model[0 if len(pos_map) == 1 else 1][key] = max(pos_map, key=pos_map.get)

    return (ngram_map, ngram_model)

In [4]:
def test_model(testing_set, model, gram_size=4, output_log_filename='output_log.txt'):
    # ---------------- STAGE 0: PREPROCESSING HISTORY ----------------
    modified = pd.DataFrame(testing_set)

    for n in range(gram_size - 1):
        modified[f'Previous POS {n + 1}'] = modified['POS'].shift(n + 1)
        modified[f'Previous POS {n + 1}'] = modified[f'Previous POS {n + 1}'].where(modified['Sentence #'] == modified['Sentence #'].shift(n + 1))

    # ---------------- STAGE 1: TESTING EACH WORD ----------------
    correct_count = 0
    def test_with_model(row):
        previous_pos = list(map(lambda n : row[f'Previous POS {n}'], list(range(1, gram_size))[::-1]))
        previous_pos = list(filter(lambda pos : not pd.isna(pos), previous_pos))
        if len(previous_pos) > 0:
            pattern = (*previous_pos, row['Word'])
        else:
            pattern = row['Word']
        # construct n grams
        grams = []
        for gram_index in range(0, gram_size):
            if len(pattern) >= gram_index+1:
                key = tuple(pattern[min(len(pattern), gram_size)-gram_index-1:]) if gram_index > 0 else pattern[len(pattern)-1]
                grams.append(key)

        # tesing against model
        expected = row['POS']
        answer = None
        # unanimous, in ascending order
        for gram in grams:
            if gram in model[0]:
                answer = model[0][gram]
                break 
        # ngrams, in descending order
        if answer is None:
            for gram in grams[::-1]:
                if gram in model[1]:
                    answer = model[1][gram]
                    break
        # failsafe
        if answer is None:
            answer = 'NN'

        # logging and counting results
        if answer == expected:
            correct_count += 1
        with open(output_log_filename, 'a', encoding='utf-8') as output_file:
            print(f'{row['Word']}\t{expected}\t{answer}\t{'MATCH' if answer == expected else 'MISMATCH'}', file=output_file)
        
    modified.apply(test_with_model, axis=1)
    return correct_count / len(testing_set)

In [5]:
def jackknife_training(raw_dataset, slice_size, gram_size=4, output_log_filename='output_log.txt'):
    # initial output file setup - need to remove file so later functions can append
    if os.path.exists(output_log_filename):
        os.remove(output_log_filename)

    sentence_count = raw_dataset['Sentence #'].max()
    slice_index = 1

    # slicing out testing sets and leaving remaining elements for training
    while slice_index < sentence_count:
        testing_set = raw_dataset[raw_dataset['Sentence #'].between(slice_index, slice_index + slice_size - 1)]
        training_set = raw_dataset[~raw_dataset['Sentence #'].between(slice_index, slice_index + slice_size - 1)]

        print(f"Training on slice ({slice_index}, {slice_index + slice_size - 1})")
        (_, model) = generate_ngram_model(training_set, gram_size)
        accuracy = test_model(testing_set, model, gram_size, output_log_filename)
        print(accuracy)

        slice_index += slice_size

In [6]:
# jackknife_training(raw_dataset, 1000)

In [7]:
testing_set = raw_dataset.loc[raw_dataset['Sentence #'].between(1, 1000)]
training_set = raw_dataset.loc[~raw_dataset['Sentence #'].between(1, 1000)]

In [8]:
(map, model) = generate_ngram_model(training_set)

In [9]:
acc = test_model(testing_set, model)
acc

TypeError: 'dict' object is not callable